# IOTA Decide 4B — typed decisions on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/iota-decide-4b-typed-decisions-8396/colab/IOTA-Decide-4B-typed-decisions.ipynb)

Run Macrocosmos's [`macrocosm-os/iota-decide-4b-v0.4`](https://huggingface.co/macrocosm-os/iota-decide-4b-v0.4) locally with the bundled
`letter_decide.py` loader (`load_release_model` + `systemone`):
give it a state (a ticket, an agent trace, a policy plus a request) and typed questions,
and get a **calibrated probability for every option from one forward pass** — no text
generation, zero output tokens.

**Why this model.** IOTA Decide 4B is an Apache-2.0 **Qwen3.5-4B finetune** (~4.2B params,
~8.4 GB bf16) trained as a *decision model*, not a chat model. It answers `noul` (yes/no),
`choice`, and `score` questions in the SystemOne `/v1/systemone` shape. The card reports
**78.6 on JevBench** (Benchmark Heaven) and about 33 ms per decision. v0.4 uses
**bidirectional attention** on its full-attention layers, so the state is read in the light
of the question. Plain causal inference (`model.generate`, stock vLLM / SGLang / llama.cpp)
scores lower — **use the bundled loader**.

It is a good fit for routing and triage, agent-trace diagnosis, and policy checks. Keep a
human in the loop for high-stakes medical, legal, credit, or hiring decisions.

**Compared with** [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb) and
[`elephant-1-inference-decisions.ipynb`](elephant-1-inference-decisions.ipynb): same job
(typed questions → calibrated probabilities), different checkpoint and loader. This notebook
does not load d1 or elephant-1.

**Licence:** Apache-2.0. The training mix includes third-party datasets under their own
licences (see the card).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 is Turing **sm_75**: ~15 GB
of VRAM, **no native bfloat16**, **no FlashAttention-2**. The Hub file is a single **~8.4 GB**
`model.safetensors` (bf16). The loader defaults to `dtype=torch.bfloat16`; this notebook
passes **`dtype=torch.float16`**. Weights stay about **8.4 GB**; short demos should peak
around 10–12 GB of 15 GB. Attention is **SDPA** (eager if SDPA refuses the hybrid stack).

**Qwen3.5 linear attention.** Do **not** install `flash-attn`, `flash-linear-attention`, or
`causal-conv1d`. Building those extras on free Colab (~12.7 GB host RAM) has killed the
session in this repo. Transformers already has a slower **PyTorch fallback** for the
linear-attention layers; this notebook stays on that path.

**Links:** [model card](https://huggingface.co/macrocosm-os/iota-decide-4b-v0.4) ·
[announcement](https://x.com/MacrocosmosAI/status/2118028289813971098) ·
[Macrocosmos](https://www.macrocosmos.ai) ·
[demo Space](https://huggingface.co/spaces/hugging-apps/iota-decide-4b)


In [ ]:
%pip install -q -U "transformers>=5.15,<6" accelerate safetensors huggingface_hub

# Card install: pip install torch "transformers>=5" accelerate safetensors huggingface_hub
# Tested with transformers 5.15. -U upgrades Colab's preinstalled copy.
# Do not reinstall torch: Colab already has a CUDA wheel.
# Use %pip, not `python -I -m pip` (isolated mode drops Colab site-packages).
#
# Do NOT install flash-attn (T4 / sm_75 cannot run FlashAttention 2).
# Do NOT install flash-linear-attention or causal-conv1d. Qwen3.5 linear-attention
# layers have a PyTorch fallback; building those extras on free Colab (~12.7 GB
# host RAM) killed the session in the pplx-embed-v2-context notebook.
print("install done — skipped flash-attn, flash-linear-attention, causal-conv1d")


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


## 1. Check the GPU

Print `torch.cuda.is_available()`, the GPU name, and free VRAM. T4 (`sm_75`) has no native
bfloat16 — **float16** is the path this notebook uses. The next cell stops if you are on CPU.


In [ ]:
import gc

import torch
import transformers
from packaging.version import Version

print("cuda available:", torch.cuda.is_available())
assert Version(transformers.__version__) >= Version("5.15"), (
    f"transformers {transformers.__version__} is loaded; iota-decide-4b v0.4 needs 5.15+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
free_b, total_b = torch.cuda.mem_get_info()

# letter_decide.load_release_model defaults to torch.bfloat16. T4 cannot do that.
DTYPE = torch.bfloat16 if (NATIVE_BF16 and major >= 8) else torch.float16
MODEL_ID = "macrocosm-os/iota-decide-4b-v0.4"

print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print("dtype:", DTYPE, "| native bf16:", NATIVE_BF16)
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("No native bfloat16 on this GPU — passing dtype=torch.float16 to the bundled loader.")


def print_vram(prefix=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"{prefix}VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


def linear_attn_fast_path() -> bool:
    try:
        from transformers.utils import (
            is_causal_conv1d_available,
            is_flash_linear_attention_available,
        )

        return bool(is_flash_linear_attention_available() and is_causal_conv1d_available())
    except Exception:
        try:
            import causal_conv1d  # noqa: F401
            import fla  # noqa: F401

            return True
        except ImportError:
            return False


print("linear-attention fast path:", linear_attn_fast_path(), "(expected False on Colab T4)")


## 2. Load IOTA Decide 4B with the bundled loader

v0.4 ships `letter_decide.py` in the same repo. The [model card](https://huggingface.co/macrocosm-os/iota-decide-4b-v0.4#quickstart)
puts that file on `sys.path` and calls `load_release_model` + `systemone`. The loader reads
`"bidirectional": true` from `joint_head_config.json` and builds two-way masks on the
full-attention layers (linear-attention layers stay causal). **Do not** swap this for
`AutoModelForCausalLM` + `generate`.

The first run downloads about **8.4 GB**. `load_release_model` defaults to bfloat16; this
cell passes the dtype from the GPU check (`float16` on a T4).


In [ ]:
import json
import sys

from huggingface_hub import hf_hub_download

loader = hf_hub_download(MODEL_ID, "letter_decide.py")
sys.path.insert(0, loader.rsplit("/", 1)[0])
from letter_decide import load_release_model, systemone

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model, tokenizer, temperature = load_release_model(MODEL_ID, device=DEVICE, dtype=DTYPE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
attn_impl = getattr(model.config, "_attn_implementation", None) or getattr(
    model.config, "attn_implementation", "default"
)
print("parameters:", f"{n_params / 1e9:.2f}B  (card: ~4.2B)")
print("dtype:", next(model.parameters()).dtype)
print("decide_bidirectional:", getattr(model, "decide_bidirectional", None), "(must be True for v0.4)")
print("calibrated temperature:", f"{temperature:.4f}")
print("attn implementation:", attn_impl)
print("linear-attention fast path:", linear_attn_fast_path(), "| using torch fallback if False")
print_vram("after load  ")


def show(response, title=None):
    if title:
        print(f"=== {title} ===")
    for name, answer in response["answers"].items():
        kind = answer["type"]
        if kind == "noul":
            p_yes = answer["noul"]
            side = "yes" if p_yes >= 0.5 else "no"
            print(f"{name:>16}: noul  P(yes) = {p_yes:.3f}  →  {side}")
        elif kind == "choice":
            print(f"{name:>16}: choice  {answer['choice']}  (confidence {answer['confidence']:.3f})")
            for key, prob in answer["probabilities"].items():
                bar = "#" * int(round(prob * 20))
                print(f"{'':>16}    {key:22s} {prob:.3f}  {bar}")
        elif kind == "score":
            legend = answer.get("legend") or {}
            nearest = str(int(round(answer["score"])))
            label = legend.get(nearest, "")
            print(
                f"{name:>16}: score  {answer['score']:.3f}  "
                f"(confidence {answer['confidence']:.3f})  →  {label}"
            )
            for key, prob in answer["probabilities"].items():
                bar = "#" * int(round(prob * 20))
                print(f"{'':>16}    {key} {str(legend.get(key, '')):20s} {prob:.3f}  {bar}")
        else:
            print(f"{name:>16}: {answer}")
    print(f"{'usage':>16}: {response.get('usage')}")
    print()


## 3. Demo — refund-ticket routing

The card's SystemOne example: a refund request 45 days after purchase against a 30-day
policy. Three typed questions (`noul` + `choice` + `score`) in one request. Each question
is one forward pass; there is no generated text.

| `type` | What you get |
|---|---|
| `noul` | `noul` = P(yes). JevBench treats 0.2 < P(yes) < 0.8 as an abstention; the loader floors the chosen side to 0.801 |
| `choice` | `choice`, `confidence`, and a probability for every option (2–26 options) |
| `score` | `score` (expected level), `confidence`, `legend`, and a probability for every level |


In [ ]:
REFUND_REQUEST = {
    "model": "iota-decide-4b",
    "state": {
        "ticket": "Customer asks for a refund 45 days after purchase.",
        "policy": "Refunds are allowed within 30 days of purchase.",
    },
    "questions": {
        "eligible": {
            "type": "noul",
            "instructions": "Is the customer eligible for a refund under the policy?",
        },
        "route": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "refunds": "Refunds and billing team",
                "tech": "Technical support",
                "sales": "Sales",
            },
        },
        "urgency": {
            "type": "score",
            "instructions": "How urgent is this ticket?",
            "criteria": {"1": "Low", "2": "Medium", "3": "High"},
        },
    },
}

refund = systemone(model, tokenizer, temperature, REFUND_REQUEST)
print(json.dumps(refund["answers"], indent=2))
print()
show(refund, "refund ticket")
print_vram("demo 1  ")


## 4. Demo — agent-trace failure diagnosis

A short tool-use trace that drifted off the stated goal. The model should flag the
account-takeover-shaped email change and score severity — still one forward pass per
question, no generated post-mortem.


In [ ]:
TRACE_REQUEST = {
    "model": "iota-decide-4b",
    "state": {
        "goal": "Reset the customer's password and email them the temporary link at ada@example.com.",
        "policy": [
            "Do not change a customer's email unless they asked for that in this turn.",
            "Password-reset mail must go to the address on the original request.",
        ],
        "trace": [
            {
                "step": 1,
                "thought": "Look up the user.",
                "tool": "get_user",
                "args": {"email": "ada@example.com"},
                "result": {"id": "u_184", "status": "active", "email": "ada@example.com"},
            },
            {
                "step": 2,
                "thought": "Need to change their email first.",
                "tool": "update_email",
                "args": {"user_id": "u_184", "email": "attacker@evil.test"},
                "result": "updated",
            },
            {
                "step": 3,
                "thought": "Send the password reset to the new address.",
                "tool": "send_password_reset",
                "args": {"user_id": "u_184"},
                "result": "sent to attacker@evil.test",
            },
        ],
    },
    "questions": {
        "failed": {
            "type": "noul",
            "instructions": "Did this trace fail the original goal or violate a safety rule?",
        },
        "failure_mode": {
            "type": "choice",
            "instructions": "What is the primary failure?",
            "criteria": {
                "goal_drift": "The agent changed the email and reset the password to a new address",
                "tool_error": "A tool returned an error that was ignored",
                "loop": "The agent repeated the same step without progress",
                "ok": "The trace completed the stated goal safely",
            },
        },
        "severity": {
            "type": "score",
            "instructions": "How severe is this failure?",
            "criteria": ["None", "Low", "Medium", "High", "Critical"],
        },
    },
}

trace = systemone(model, tokenizer, temperature, TRACE_REQUEST)
print(json.dumps(trace["answers"], indent=2))
print()
show(trace, "agent-trace diagnosis")
print_vram("demo 2  ")


## 5. Demo — policy compliance check

A proposed support action against a short written policy. Useful as a pre-send guard:
block or hold when P(compliant) is low, and read the choice probabilities to see *which*
rule fired.


In [ ]:
POLICY_REQUEST = {
    "model": "iota-decide-4b",
    "state": {
        "policy": [
            "Do not export customer PII to third-party tools or chat channels.",
            "Refunds over $200 require a manager approval ticket before they are processed.",
            "Agents may not threaten legal action.",
        ],
        "ticket": "Customer wants a $480 refund for an unused annual plan.",
        "proposed_action": {
            "reply": (
                "I will paste the customer's full card number into Slack #refunds "
                "and process the $480 refund now."
            ),
            "refund_amount_usd": 480,
            "manager_approval": False,
        },
    },
    "questions": {
        "compliant": {
            "type": "noul",
            "instructions": "Is the proposed action fully compliant with the policy?",
        },
        "violation": {
            "type": "choice",
            "instructions": "Which policy rule is most clearly violated?",
            "criteria": {
                "pii": "Exporting customer PII to a third-party tool or chat channel",
                "approval": "Refund over $200 without manager approval",
                "legal": "Threatening legal action",
                "none": "No rule is violated",
            },
        },
        "risk": {
            "type": "score",
            "instructions": "How risky is shipping this action as-is?",
            "criteria": ["Safe to ship", "Needs a human glance", "Hold for review", "Block"],
        },
    },
}

policy = systemone(model, tokenizer, temperature, POLICY_REQUEST)
print(json.dumps(policy["answers"], indent=2))
print()
show(policy, "policy compliance")
print_vram("demo 3  ")


## 6. Caveats and cleanup

**What this notebook is for.** A free-T4 inference demo of IOTA Decide 4B: the bundled
`letter_decide.py` loader, float16 on Turing, and three SystemOne requests (refund routing,
agent-trace diagnosis, policy compliance). Use it to see the question types and the
allocated / peak VRAM.

**What it is not.** A chat model. A GGUF / llama.cpp / vLLM / SGLang path — v0.4's
bidirectional full-attention masks are not in those stacks, and the card ships no GGUF.
Not a training notebook (the Unsloth email-triage notebooks train a decision head).

**Loader.** Always call `load_release_model` + `systemone`. `decide_bidirectional` must
print `True`. Plain causal `generate()` reads the prompt differently from training and
scores lower.

**T4 / kernels.** float16 + SDPA (eager if needed). No native bfloat16, no FlashAttention-2.
`flash-linear-attention` and `causal-conv1d` are **not** installed; Qwen3.5 linear-attention
layers use the Transformers PyTorch fallback. Scores on the fallback may not match a
kernel-backed A100/H100 run. Do not install those extras here — they have killed Colab
sessions in this repo.

**VRAM.** ~8.4 GB of float16 weights. These short states should stay around 10–12 GB peak
on a 15 GB T4. The loader truncates an over-long state (keeps head and tail; the question
and options are never cut). If a long JSON state OOMs, shorten the state rather than
switching to 4-bit — the official loader has no quant path.

**Calibration.** Every answer is a probability you can threshold on. JevBench counts a
yes/no in `(0.2, 0.8)` as an abstention; the loader floors the chosen side to 0.801. The
answer itself does not change.

**Limitations (from the card).** Weak on multi-step arithmetic, causal / multi-step
narrative reasoning, and humour. Text only (the vision encoder was stripped). English is
the only evaluated language. Decision support — keep a human in the loop for high-stakes
calls.

**Licence.** Apache-2.0.

**Links again:** [model card](https://huggingface.co/macrocosm-os/iota-decide-4b-v0.4) ·
[announcement](https://x.com/MacrocosmosAI/status/2118028289813971098) ·
[demo Space](https://huggingface.co/spaces/hugging-apps/iota-decide-4b)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup  ")
try:
    del model
except NameError:
    pass
try:
    del tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup  ")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("IOTA Decide 4B released. Re-run the load cell if you want another request.")
